In [2]:
import os
import pandas as pd
import numpy as np

print("\n--- 第一步：读取原始数据 ---")
df = pd.read_csv('../data/raw/LENDHIST2019_2020_utf8.csv', encoding_errors='ignore')
total_records = df.shape[0]
print(f"读取完毕！原始数据量: {total_records} 条记录, {df.shape[1]} 个字段")

# 【前置处理】：将特殊的字符串 'nan' 替换为真实的 numpy 缺失值，以便后续精准统计
df = df.replace('nan', np.nan)
df['LEND_DATE'] = df['LEND_DATE'].replace('nan', np.nan)
df['RET_DATE'] = df['RET_DATE'].replace('nan', np.nan)

print("\n--- 数据质量审查：缺失值统计分析 ---")
check_cols = ['DEPT', 'OCCUPATION', 'ABSTRACT', 'SUB', 'RET_DATE']
print(f"{'字段名称':<15} | {'缺失数量':<10} | {'缺失比例'}")
print("-" * 45)
for col in check_cols:
    if col in df.columns:
        missing_count = df[col].isnull().sum()
        # 针对文本字段，把空字符串也算作缺失
        if df[col].dtype == object:
            missing_count += (df[col].astype(str).str.strip() == '').sum()

        missing_rate = (missing_count / total_records) * 100
        print(f"{col:<15} | {missing_count:<10} | {missing_rate:.2f}%")


print("\n--- 第二步：缺失值填补 ---")
# 1. 用户画像维度：填充未知信息
user_cols = ['SEX', 'DEPT', 'OCCUPATION']
for col in user_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

# 年级缺失通常用 0 或一个特定数字占位
if 'CODE01' in df.columns:
    df['CODE01'] = pd.to_numeric(df['CODE01'], errors='coerce').fillna(0)

# 2. 图书属性维度：填充空字符串
book_cols = ['ABSTRACT', 'SUB', 'CALL_NO', 'AUTHOR', 'PUBLISHER', 'AU']
for col in book_cols:
    if col in df.columns:
        df[col] = df[col].fillna('')
print("文本与分类特征缺失值填补完毕。")

print("\n--- 第三步：时间格式转换与特征构建 ---")
# 修复特殊的无空格时间格式 (例如 '2020-01-1321:23:17' 转换为 '2020-01-13 21:23:17')
df['LEND_DATE'] = df['LEND_DATE'].astype(str).str.replace(r'(\d{4}-\d{2}-\d{2})(\d{2}:\d{2}:\d{2})', r'\1 \2', regex=True)
df['RET_DATE'] = df['RET_DATE'].astype(str).str.replace(r'(\d{4}-\d{2}-\d{2})(\d{2}:\d{2}:\d{2})', r'\1 \2', regex=True)

# 转换为 datetime 对象
df['LEND_DATE'] = pd.to_datetime(df['LEND_DATE'], errors='coerce')
df['RET_DATE'] = pd.to_datetime(df['RET_DATE'], errors='coerce')

# 构建新特征：是否已归还 (1=是, 0=否)
df['IS_RETURNED'] = df['RET_DATE'].notnull().astype(int)

# 构建新特征：借阅时长(天)。未归还的保留为空 (NaN)
df['LEND_DAYS'] = (df['RET_DATE'] - df['LEND_DATE']).dt.total_seconds() / (24 * 3600)

# 构建新特征：借阅行为时间点
df['LEND_MONTH'] = df['LEND_DATE'].dt.month
df['LEND_HOUR'] = df['LEND_DATE'].dt.hour
df['LEND_DAYOFWEEK'] = df['LEND_DATE'].dt.dayofweek + 1


print("\n--- 数据质量审查：异常值统计分析 ---")
# 统计三种典型异常的数量
# 异常 1：借阅时间无效或缺失
invalid_date_count = df['LEND_DATE'].isnull().sum()
# 异常 2：秒借秒还（有归还记录，但借阅时长 <= 0）
quick_return_count = ((df['IS_RETURNED'] == 1) & (df['LEND_DAYS'] <= 0)).sum()
# 异常 3：重复打点（同一用户在同一时间对同一本书的记录）
dup_count = df.duplicated(subset=['USERID', 'BOOK_ID', 'LEND_DATE'], keep='first').sum()

print(f"发现异常情况 1 [借阅时间无效/缺失] : {invalid_date_count} 条")
print(f"发现异常情况 2 [借阅时长<=0秒借秒还] : {quick_return_count} 条")
print(f"发现异常情况 3 [同一借阅行为重复记录] : {dup_count} 条")


print("\n--- 第四步：异常值清洗执行 ---")
# 规则1：必须有有效的借阅时间
mask_valid_date = df['LEND_DATE'].notnull()

# 规则2：如果图书已归还，借阅时长必须大于0
mask_valid_duration = (df['IS_RETURNED'] == 0) | (df['LEND_DAYS'] > 0)

# 应用过滤规则
df_cleaned = df[mask_valid_date & mask_valid_duration].copy()

# 规则3：剔除重复数据
df_cleaned = df_cleaned.drop_duplicates(subset=['USERID', 'BOOK_ID', 'LEND_DATE'], keep='first')

print(f"异常值清洗完毕！")
print(f"清洗前总数: {total_records} 条")
print(f"最终保留有效数据: {df_cleaned.shape[0]} 条")
print(f"共计清理脏数据: {total_records - df_cleaned.shape[0]} 条")

print("\n--- 第五步：保存高质量数据 ---")
os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/LENDHIST2019_2020_cleaned.csv'
df_cleaned.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"数据已安全落盘: {output_path}")

# 预览前5行
display(df_cleaned.head())


--- 第一步：读取原始数据 ---
读取完毕！原始数据量: 99254 条记录, 22 个字段

--- 数据质量审查：缺失值统计分析 ---
字段名称            | 缺失数量       | 缺失比例
---------------------------------------------
DEPT            | 0          | 0.00%
OCCUPATION      | 8129       | 8.19%
ABSTRACT        | 14643      | 14.75%
SUB             | 1091       | 1.10%
RET_DATE        | 5532       | 5.57%

--- 第二步：缺失值填补 ---
文本与分类特征缺失值填补完毕。

--- 第三步：时间格式转换与特征构建 ---

--- 数据质量审查：异常值统计分析 ---
发现异常情况 1 [借阅时间无效/缺失] : 0 条
发现异常情况 2 [借阅时长<=0秒借秒还] : 586 条
发现异常情况 3 [同一借阅行为重复记录] : 0 条

--- 第四步：异常值清洗执行 ---
异常值清洗完毕！
清洗前总数: 99254 条
最终保留有效数据: 98668 条
共计清理脏数据: 586 条

--- 第五步：保存高质量数据 ---
数据已安全落盘: ../data/processed/LENDHIST2019_2020_cleaned.csv


,USERID,BIRTHYEAR,SEX,DEPT,OCCUPATION,CODE01,REDR_TYPE_NAME,LEND_DATE,RET_DATE,RENEW_TIMES,...,PUBLISHER,ISBN,PUB_YEAR,LOCATION_NAME,DOC_TYPE_NAME,IS_RETURNED,LEND_DAYS,LEND_MONTH,LEND_HOUR,LEND_DAYOFWEEK
0,b1e9675eeacb2d6edcf031bdac1bf0df,1990.0,女,应用金融与行为科学学院,行为经济学,2013,研究生,2020-01-13 21:23:17,2020-06-16 09:03:35,0,...,经济科学出版社,7-5058-1613-6,1998.0,辅助书库一,中文图书,1,154.486319,1,21,1
1,860b7ac7a8a12790da3d77d53e265a21,1998.0,女,公共管理学院,公共管理类(含公共事业管理、行政管理和劳动与社会保障),2018,本科生,2020-12-15 18:10:22,NaT,0,...,武汉大学出版社,7-307-02726-7,1999.0,北区六层,中文图书,0,NaN,12,18,2
2,5a9b0d6f361f4832a93db1ba02a73adc,1996.0,女,会计学院,会计,2018,研究生,2019-11-29 09:46:42,2019-12-30 13:51:18,0,...,西南财经大学出版社,7-81055-433-6,1999.0,北区五层,中文图书,1,31.169861,11,9,5
3,ca2bc2cb7fc2c5e66075aa6b6387c123,1989.0,男,经济学院,产业经济学,2016,研究生,2019-05-17 16:11:00,2019-07-09 16:34:40,0,...,中国人民大学出版社,7-300-02380-0,1997.0,北区四层,中文图书,1,53.016435,5,16,5
4,dca2a9c89a64cbfb95506b69570b9cb6,1996.0,女,法学院,民商法学,2020,研究生,2020-12-28 10:32:48,NaT,0,...,生活.读书.新知三联书店,7-108-00663-4,1994.0,北区二层,中文图书,0,NaN,12,10,1
